# 07 — Resampling and Validation
**Goal:** Master the cross-validation and bootstrap machinery that every
later chapter (regularization, model selection, ensembling) assumes you
know. Source: ISLR Ch5, Efron & Tibshirani (1993) *An Introduction to the
Bootstrap*.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/learning_courses')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (KFold, LeaveOneOut, LeavePOut,
                                     ShuffleSplit, StratifiedKFold,
                                     cross_val_score, cross_val_predict,
                                     learning_curve, validation_curve)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
np.random.seed(7)

## 1. Why we need resampling

Training error is **biased** — it systematically underestimates the true
risk. We need an estimate of 𝔼[loss] on data the model did *not* see.

Two families of methods:

- **Cross-validation** — partition the data; train on k-1 folds, test on the
  kth; average.
- **Bootstrap** — sample *with replacement* of size n; compute the
  statistic on each sample; use the distribution to estimate variance and
  bias.

Both answer the same question: *how would this model do on data it has not
seen?*

## 2. The validation set approach

Random split into train and test. The simplest approach, with two
weaknesses:

1. **High variance** — different random splits give different error estimates.
2. **Wastes data** — the test set is not used for training.

Use it for quick exploration only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
np.random.seed(0)
n = 100
X = np.linspace(0, 6, n).reshape(-1, 1)
y = np.sin(1.5 * X.ravel()) + 0.3 * np.random.randn(n)
errors = []
for seed in range(20):
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.3, random_state=seed)
    m = make_pipeline(PolynomialFeatures(5), LinearRegression()).fit(Xt, yt)
    errors.append(mean_squared_error(yv, m.predict(Xv)))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(range(len(errors)), errors, color='steelblue')
ax.axhline(np.mean(errors), color='crimson', lw=2, label=f'mean = {np.mean(errors):.3f}')
ax.set_xlabel('split seed'); ax.set_ylabel('test MSE')
ax.set_title('Validation set approach: high variance across splits')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 3. K-fold cross-validation

Split the data into k equally-sized folds. Train on k-1, test on the held-out
fold. Repeat k times so every example is in the test fold exactly once. The
estimate is the average.

**Typical choices:** k = 5 or k = 10. Larger k → lower bias, higher variance
(and more computation). k = n is **leave-one-out** (LOOCV) — nearly unbiased
but with high variance because the training sets are nearly identical.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
n, k = 25, 5
rng = np.random.default_rng(0)
perm = rng.permutation(n)
fold_size = n // k
colors = plt.cm.tab10(np.arange(k))
for fold in range(k):
        for j in range(k):
            if j == fold:
                test_idx = perm[j*fold_size:(j+1)*fold_size]
            else:
                pass
for i in range(n):
    for fold in range(k):
        in_test = (perm[fold*fold_size:(fold+1)*fold_size] == i).any()
        ax.add_patch(plt.Rectangle((fold, i), 1, 1, color='crimson' if in_test else 'steelblue', alpha=0.7))
ax.set_xticks(np.arange(k) + 0.5); ax.set_xticklabels([f'fold {j}' for j in range(k)])
ax.set_yticks(np.arange(0, n, 5) + 0.5); ax.set_yticklabels(np.arange(0, n, 5))
ax.set_xlim(0, k); ax.set_ylim(0, n)
ax.set_title('5-fold CV — red = test, blue = train (each example in test exactly once)')
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data[:1000], data.target[:1000]

for k in [2, 5, 10, len(X)]:
    if k == len(X):
        cv = LeaveOneOut()
        name = 'LOOCV'
    else:
        cv = KFold(n_splits=k, shuffle=True, random_state=0)
        name = f'{k}-fold'
    scores = cross_val_score(LinearRegression(), X, y, cv=cv, scoring='neg_mean_squared_error')
    print(f'{name:8s}  MSE = {-scores.mean():.4f} \u00b1 {-scores.std():.4f}  ({len(scores)} fits)')

## 4. Stratified K-fold

For classification, each fold should have approximately the same class
proportions as the full data. `StratifiedKFold` does this automatically.
Use it for any classification CV; for regression with rare target values,
use `StratifiedKFold` on binned targets.

In [ ]:
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer().data, load_breast_cancer().target
for cv, name in [(KFold(5, shuffle=True, random_state=0), 'K-Fold'),
                (StratifiedKFold(5, shuffle=True, random_state=0), 'Stratified')]:
    props = []
    for _, idx in cv.split(X, y):
        props.append(y[idx].mean())
    print(f'{name:12s}  class-1 fraction per fold: {[round(p,3) for p in props]}')

## 5. The one-standard-error rule

When picking between models of different complexity, the *smallest* model
whose CV error is within one standard error of the minimum is a good
choice. This is the **1-SE rule** (Breiman, Spector, Subrahmanian 1994) and
biases you toward simpler models without sacrificing much accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score
degrees = list(range(1, 11))
means, stds = [], []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression())
    s = -cross_val_score(m, X, y, cv=5, scoring='neg_mean_squared_error')
    means.append(s.mean()); stds.append(s.std())
means, stds = np.array(means), np.array(stds)
best = np.argmin(means)
one_se_threshold = means[best] + stds[best]
simplest = max(d for d, m in zip(degrees, means) if m <= one_se_threshold)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(degrees, means, yerr=stds, marker='o', color='steelblue', lw=2, capsize=3)
ax.axvline(simplest, color='seagreen', linestyle='--', label=f'1-SE rule picks d={simplest}')
ax.set_xlabel('polynomial degree'); ax.set_ylabel('CV MSE')
ax.set_title('1-SE rule — pick simplest model within 1 SE of the minimum')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 6. The bias-variance of CV

The CV error is itself a random variable (it depends on the random splits).
It has a bias and a variance. The bias is *upward* — CV slightly
overestimates the true risk because each training set is smaller than the
full set.

**LOOCV** has the lowest bias but the highest variance because the n
training sets are nearly identical. **K-fold with k=5 or 10** is a good
practical compromise.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
kvals = [2, 3, 5, 10, 20, 50, n if False else 100]  # last is approx LOOCV
bias, var = [], []
for k in kvals:
    if k >= n: continue
    scores = [-cross_val_score(LinearRegression(), X, y, cv=KFold(k, shuffle=True, random_state=s), scoring='neg_mean_squared_error').mean() for s in range(10)]
    bias.append(np.mean(scores) - np.mean(scores))
    var.append(np.var(scores))
ax.plot(kvals[:len(bias)], var, marker='o', color='crimson', label='variance of CV estimate')
ax.set_xlabel('k (number of folds)'); ax.set_ylabel('variance')
ax.set_title('Variance of CV estimate grows as k grows')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 7. The bootstrap

**Idea:** sample n observations with replacement, compute the statistic on
the sample, repeat B times. The empirical distribution of the statistic is
an estimate of its sampling distribution.

**Why it works:** with replacement, each bootstrap sample is missing about
37% of the original data on average (1 - 1/e ≈ 0.632).

**Use cases:**
1. Estimate the standard error of an estimator that has no formula.
2. Construct confidence intervals (percentile, BCa, t-bootstrap).
3. Bagging (notebook 09) — train a model on each bootstrap sample and
   average.

In [ ]:
def bootstrap_statistic(data, statistic_fn, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(data)
    stats = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        stats[b] = statistic_fn(data[idx])
    return stats

rng = np.random.default_rng(0)
data = rng.normal(0, 1, 30)
true_se = data.std(ddof=1) / np.sqrt(len(data))
boot = bootstrap_statistic(data, np.mean, B=5000, seed=0)
print(f'true SE of mean  : {true_se:.4f}')
print(f'bootstrap SE     : {boot.std():.4f}')

## 8. Bootstrap confidence intervals

Three flavors, in increasing sophistication:

1. **Normal approximation** — θ̂ ± z · SE_boot.
2. **Percentile** — (α/2, 1-α/2) quantiles of the bootstrap distribution.
3. **Pivotal** — 2θ̂ − (bootstrap quantile at 1-α/2, α/2).
4. **BCa** — bias-corrected and accelerated; the most accurate in practice.

For most ML applications the percentile interval is enough.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot, bins=50, color='steelblue', edgecolor='white', density=True)
lo, hi = np.percentile(boot, [2.5, 97.5])
ax.axvline(lo, color='crimson', lw=2, label=f'95% percentile CI: ({lo:.3f}, {hi:.3f})')
ax.axvline(hi, color='crimson', lw=2)
ax.set_xlabel('bootstrap mean'); ax.set_ylabel('density')
ax.set_title('Bootstrap distribution of the sample mean')
ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 9. Learning curves — is more data the answer?

`learning_curve` trains the model on growing subsets and reports train and
validation error. Two diagnostic patterns:

- **High bias:** train and validation error both high and *close together*.
  Adding data does not help — you need a richer model or better features.
- **High variance:** train error low, validation error much higher, and a
  big gap. Adding data *will* help — or regularize more.

In [ ]:
from sklearn.model_selection import learning_curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, d, name in zip(axes, [1, 15], ['high-bias', 'high-variance']):
    train_sizes, tr, va = learning_curve(
        make_pipeline(PolynomialFeatures(d), LinearRegression()),
        X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='neg_mean_squared_error')
    tr = -tr.mean(axis=1); va = -va.mean(axis=1)
    ax.plot(train_sizes, tr, marker='o', color='steelblue', label='train')
    ax.plot(train_sizes, va, marker='s', color='crimson',  label='val')
    ax.set_title(f'degree {d} ({name})')
    ax.set_xlabel('n train'); ax.set_ylabel('MSE')
    ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

## 10. Nested CV — the *honest* hyperparameter search

If you use the same CV to *both* pick hyperparameters and report
performance, the reported error is biased downward. The fix is **nested CV**:
an outer loop that estimates generalization, and an inner loop that picks
hyperparameters on each training fold.

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
param_grid = {'polynomialfeatures__degree': [1, 2, 3, 5, 8]}
inner = GridSearchCV(make_pipeline(PolynomialFeatures(), LinearRegression()),
                     param_grid, cv=KFold(5, shuffle=True, random_state=0))
outer_scores = cross_val_score(inner, X, y, cv=KFold(5, shuffle=True, random_state=1),
                               scoring='neg_mean_squared_error')
print(f'Nested CV MSE: {-outer_scores.mean():.4f} \u00b1 {-outer_scores.std():.4f}')

## Summary

| Method | Use |
|---|---|
| Train/val/test | quick exploration; never use test for tuning |
| 5- or 10-fold CV | default for model selection |
| LOOCV | small n, but high variance |
| 1-SE rule | pick simplest model within 1 SE of min |
| Bootstrap | estimate SE, build CIs, bagging |
| Nested CV | honest hyperparameter selection |
| Learning curves | diagnose bias vs variance |

**Next:** `08_regularization_and_selection.ipynb` — make a rich model
behave.